In [2]:
from __future__ import annotations

import ast
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Mapping, Sequence, Set, Tuple

import pandas as pd


In [3]:
def normalize_identifier(name: str) -> str:
    """Normalize table / column names without changing letter case."""
    return name.strip()


def parse_string_list(raw_value: Any) -> List[str]:
    """Convert a list-like field into a cleaned list of strings."""
    if raw_value is None:
        return []

    if isinstance(raw_value, list):
        return [normalize_identifier(str(item)) for item in raw_value if str(item).strip()]

    if isinstance(raw_value, str):
        text: str = raw_value.strip()
        if not text:
            return []

        for parser in (json.loads, ast.literal_eval):
            try:
                parsed: Any = parser(text)
                if isinstance(parsed, list):
                    return [normalize_identifier(str(item)) for item in parsed if str(item).strip()]
            except (ValueError, SyntaxError, json.JSONDecodeError):
                continue

        return [normalize_identifier(part) for part in text.split(",") if part.strip()]

    return [normalize_identifier(str(raw_value))]


def parse_columns_by_table(raw_value: Any) -> Dict[str, Set[str]]:
    """Parse either a JSON-like dict string or the ground-truth `table: [col1, col2]` format."""
    if raw_value is None:
        return {}

    parsed_mapping: Dict[str, Any] | None = None

    if isinstance(raw_value, Mapping):
        parsed_mapping = dict(raw_value)
    elif isinstance(raw_value, str):
        text: str = raw_value.strip()
        if not text:
            return {}

        for parser in (json.loads, ast.literal_eval):
            try:
                candidate: Any = parser(text)
                if isinstance(candidate, dict):
                    parsed_mapping = candidate
                    break
            except (ValueError, SyntaxError, json.JSONDecodeError):
                continue

        if parsed_mapping is None:
            parsed_mapping = {}
            # Ground truth uses: `table: [col1, col2]; another_table: [col3]`
            for match in re.finditer(r"([^:;]+):\s*\[(.*?)\]", text):
                table_name: str = normalize_identifier(match.group(1))
                raw_columns: str = match.group(2)
                column_list: List[str] = [
                    normalize_identifier(column)
                    for column in raw_columns.split(",")
                    if column.strip()
                ]
                parsed_mapping[table_name] = column_list
    else:
        return {}

    normalized_mapping: Dict[str, Set[str]] = {}
    for table_name, columns in parsed_mapping.items():
        normalized_table: str = normalize_identifier(str(table_name))
        if isinstance(columns, (list, tuple, set)):
            normalized_columns: Set[str] = {
                normalize_identifier(str(column))
                for column in columns
                if str(column).strip()
            }
        elif columns is None:
            normalized_columns = set()
        else:
            normalized_columns = {
                normalize_identifier(str(columns))
            } if str(columns).strip() else set()

        normalized_mapping[normalized_table] = normalized_columns

    return normalized_mapping


def parse_prediction_record(record: Mapping[str, Any]) -> Tuple[Set[str], Dict[str, Set[str]]]:
    """Parse one prediction item and return predicted tables plus predicted columns-by-table."""
    if "response" in record:
        columns_by_table: Dict[str, Set[str]] = parse_columns_by_table(record.get("response"))
        predicted_tables: Set[str] = set(columns_by_table.keys())
        return predicted_tables, columns_by_table

    predicted_tables = set(parse_string_list(record.get("relevant_tables")))
    columns_by_table = parse_columns_by_table(record.get("relevant_columns"))
    return predicted_tables, columns_by_table


In [4]:
def build_example_id(dataset_tag: str, local_id: int) -> str:
    """Match the id format used in the log files, for example `MMQA-two_table-0`."""
    return f"MMQA-{dataset_tag}-{local_id}"


def infer_dataset_tag(file_path: Path) -> str:
    """Infer whether a ground-truth file belongs to the two-table or three-table split."""
    file_name: str = file_path.name.lower()
    if "two_table" in file_name:
        return "two_table"
    if "three_table" in file_name:
        return "three_table"
    raise ValueError(f"Cannot infer dataset tag from: {file_path}")


def load_ground_truth_records(file_paths: Sequence[Path]) -> Dict[str, Dict[str, Any]]:
    """Load and normalize all labels, keyed by the same id used in prediction logs."""
    ground_truth: Dict[str, Dict[str, Any]] = {}

    for file_path in file_paths:
        dataset_tag: str = infer_dataset_tag(file_path)
        rows: List[Dict[str, Any]] = json.loads(file_path.read_text())

        for row in rows:
            example_id: str = build_example_id(dataset_tag, int(row["id_"]))
            columns_by_table: Dict[str, Set[str]] = parse_columns_by_table(row.get("sql_columns_by_table"))
            ground_truth[example_id] = {
                "split": dataset_tag,
                "tables": set(columns_by_table.keys()),
                "columns_by_table": columns_by_table,
                "question": row.get("Question", ""),
                "spider_db_id": row.get("Spider_db_id", ""),
            }

    return ground_truth


def load_prediction_records(file_path: Path) -> Dict[str, Dict[str, Any]]:
    """Load one prediction file and normalize it into the common structure."""
    rows: List[Dict[str, Any]] = json.loads(file_path.read_text())
    predictions: Dict[str, Dict[str, Any]] = {}

    for row in rows:
        predicted_tables, columns_by_table = parse_prediction_record(row)
        predictions[row["id"]] = {
            "tables": predicted_tables,
            "columns_by_table": columns_by_table,
            "question": row.get("question", ""),
            "spider_db_id": row.get("spider_db_id", ""),
        }

    return predictions


def infer_model_name(prediction_file: Path, log_root: Path) -> str:
    """Use the first folder under LOG_ROOT as the model name; root-level files map to `root`."""
    try:
        relative_path: Path = prediction_file.relative_to(log_root)
    except ValueError:
        return "external"

    if len(relative_path.parts) == 1:
        return "root"

    return relative_path.parts[0]


def infer_shot_setting(prediction_file: Path) -> str:
    """Infer whether a file belongs to the zero-shot or few-shot group."""
    file_name: str = prediction_file.stem.lower()
    if "few_shot" in file_name:
        return "few_shot"
    if "zero_shot" in file_name:
        return "zero_shot"
    return "unknown"


def infer_method_name(prediction_file: Path) -> str:
    """Infer which schema-linking method produced this file."""
    file_name: str = prediction_file.stem.lower()
    if "table_column" in file_name:
        return "table_column"
    return "baseline"


def build_prediction_label(prediction_file: Path, log_root: Path) -> str:
    """Show a stable relative path so same-named files from different models stay distinguishable."""
    try:
        return prediction_file.relative_to(log_root).as_posix()
    except ValueError:
        return prediction_file.name


def include_value(value: str, selected_values: Sequence[str]) -> bool:
    """Treat an empty selection list as `include all`."""
    normalized_values: Set[str] = {
        item.strip()
        for item in selected_values
        if item.strip()
    }
    if not normalized_values:
        return True
    return value in normalized_values


def collect_prediction_entries(
    log_root: Path,
    selected_models: Sequence[str],
    selected_shot_settings: Sequence[str],
    selected_methods: Sequence[str],
) -> List[Dict[str, Any]]:
    """Collect prediction files and annotate them with model / shot / method metadata."""
    prediction_entries: List[Dict[str, Any]] = []

    for prediction_file in sorted(log_root.rglob("*.json")):
        model_name: str = infer_model_name(prediction_file, log_root)
        shot_setting: str = infer_shot_setting(prediction_file)
        method_name: str = infer_method_name(prediction_file)

        if not include_value(model_name, selected_models):
            continue
        if not include_value(shot_setting, selected_shot_settings):
            continue
        if not include_value(method_name, selected_methods):
            continue

        prediction_entries.append(
            {
                "path": prediction_file,
                "model": model_name,
                "shot_setting": shot_setting,
                "method": method_name,
                "prediction_name": prediction_file.stem,
                "prediction_file": build_prediction_label(prediction_file, log_root),
            }
        )

    return prediction_entries


def filter_ground_truth_by_split(
    ground_truth: Mapping[str, Dict[str, Any]],
    dataset_split: str,
) -> Dict[str, Dict[str, Any]]:
    """Keep only examples from one split so metrics can be reported separately."""
    return {
        example_id: record
        for example_id, record in ground_truth.items()
        if record.get("split") == dataset_split
    }


In [5]:
def flatten_column_pairs(columns_by_table: Mapping[str, Set[str]]) -> Set[Tuple[str, str]]:
    """Flatten `{table: {columns}}` into `(table, column)` pairs for column-level scoring."""
    flattened_pairs: Set[Tuple[str, str]] = set()
    for table_name, columns in columns_by_table.items():
        for column_name in columns:
            flattened_pairs.add((table_name, column_name))
    return flattened_pairs


def compute_prf(tp: int, fp: int, fn: int) -> Dict[str, float]:
    """Compute precision / recall / F1 from counts, guarding against divide-by-zero."""
    precision: float = tp / (tp + fp) if (tp + fp) else 0.0
    recall: float = tp / (tp + fn) if (tp + fn) else 0.0
    f1: float = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def evaluate_predictions(
    ground_truth: Mapping[str, Dict[str, Any]],
    predictions: Mapping[str, Dict[str, Any]],
) -> Tuple[pd.DataFrame, Dict[str, float], Dict[str, float]]:
    """Run micro-averaged evaluation for table retrieval and column retrieval."""
    detail_rows: List[Dict[str, Any]] = []
    table_tp = table_fp = table_fn = 0
    column_tp = column_fp = column_fn = 0

    for example_id, gold_record in ground_truth.items():
        predicted_record: Dict[str, Any] = dict(predictions.get(example_id, {}))

        gold_tables: Set[str] = set(gold_record["tables"])
        pred_tables: Set[str] = set(predicted_record.get("tables", set()))

        gold_column_pairs: Set[Tuple[str, str]] = flatten_column_pairs(gold_record["columns_by_table"])
        pred_column_pairs: Set[Tuple[str, str]] = flatten_column_pairs(predicted_record.get("columns_by_table", {}))

        current_table_tp: int = len(gold_tables & pred_tables)
        current_table_fp: int = len(pred_tables - gold_tables)
        current_table_fn: int = len(gold_tables - pred_tables)

        current_column_tp: int = len(gold_column_pairs & pred_column_pairs)
        current_column_fp: int = len(pred_column_pairs - gold_column_pairs)
        current_column_fn: int = len(gold_column_pairs - pred_column_pairs)

        table_tp += current_table_tp
        table_fp += current_table_fp
        table_fn += current_table_fn
        column_tp += current_column_tp
        column_fp += current_column_fp
        column_fn += current_column_fn

        detail_rows.append(
            {
                "id": example_id,
                "table_tp": current_table_tp,
                "table_fp": current_table_fp,
                "table_fn": current_table_fn,
                "column_tp": current_column_tp,
                "column_fp": current_column_fp,
                "column_fn": current_column_fn,
            }
        )

    detail_df: pd.DataFrame = pd.DataFrame(detail_rows)
    table_scores: Dict[str, float] = compute_prf(table_tp, table_fp, table_fn)
    column_scores: Dict[str, float] = compute_prf(column_tp, column_fp, column_fn)

    return detail_df, table_scores, column_scores


In [6]:
# Ground-truth files. The filenames on disk use `_db_id`, so we point to the real paths here.
GROUND_TRUTH_FILES: List[Path] = [
    Path("/home/xubeiyu/projects/Schemalinking/Data/MMQA/Synthesized_two_table_with_spider_db_id.json"),
    Path("/home/xubeiyu/projects/Schemalinking/Data/MMQA/Synthesized_three_table_with_spider_db_id.json"),
]

LOG_ROOT: Path = Path("/home/xubeiyu/projects/Schemalinking/Logs")

# Leave empty to evaluate every model folder under LOG_ROOT.
SELECTED_MODELS: List[str] = ["gpt-5-mini", "gpt-4o-mini"]
# Example: SELECTED_MODELS = ["gpt-5-mini", "gpt-4o-mini"]

# Leave empty to include both `zero_shot` and `few_shot`.
SELECTED_SHOT_SETTINGS: List[str] = ["few_shot","zero_shot"]
# Example: SELECTED_SHOT_SETTINGS = ["few_shot"]

# Leave empty to include both methods: `baseline` and `table_column`.
SELECTED_METHODS: List[str] = ["baseline", "table_column"]
# Example: SELECTED_METHODS = ["baseline", "table_column"]

DATASET_SPLITS: List[str] = ["two_table", "three_table"]


In [7]:
ground_truth_records: Dict[str, Dict[str, Any]] = load_ground_truth_records(GROUND_TRUTH_FILES)
prediction_entries: List[Dict[str, Any]] = collect_prediction_entries(
    LOG_ROOT,
    SELECTED_MODELS,
    SELECTED_SHOT_SETTINGS,
    SELECTED_METHODS,
)

if not prediction_entries:
    raise ValueError(
        f"No prediction files matched under {LOG_ROOT}. "
        "Adjust SELECTED_MODELS, SELECTED_SHOT_SETTINGS, or SELECTED_METHODS."
    )

PREDICTION_FILES: List[Path] = [entry["path"] for entry in prediction_entries]
prediction_file_catalog: pd.DataFrame = pd.DataFrame(prediction_entries).drop(columns=["path"])
prediction_file_catalog = prediction_file_catalog.sort_values(
    ["model", "shot_setting", "method", "prediction_file"]
).reset_index(drop=True)


In [8]:
print(f"Loaded {len(ground_truth_records)} labeled examples.")
print(f"Matched {len(prediction_entries)} prediction files from {LOG_ROOT}.")
prediction_file_catalog


Loaded 3313 labeled examples.
Matched 8 prediction files from /home/xubeiyu/projects/Schemalinking/Logs.


,model,shot_setting,method,prediction_name,prediction_file
0,gpt-4o-mini,few_shot,baseline,few_shot_baseline_schema_linking_MMQA_20260303...,gpt-4o-mini/few_shot_baseline_schema_linking_M...
1,gpt-4o-mini,few_shot,table_column,few_shot_table_column_MMQA_20260303_233424,gpt-4o-mini/few_shot_table_column_MMQA_2026030...
2,gpt-4o-mini,zero_shot,baseline,zero_shot_baseline_schema_linking_MMQA_2026030...,gpt-4o-mini/zero_shot_baseline_schema_linking_...
3,gpt-4o-mini,zero_shot,table_column,zero_shot_table_column_MMQA_20260303_233819,gpt-4o-mini/zero_shot_table_column_MMQA_202603...
4,gpt-5-mini,few_shot,baseline,few_shot_baseline_schema_linking_MMQA_20260302...,gpt-5-mini/few_shot_baseline_schema_linking_MM...
5,gpt-5-mini,few_shot,table_column,few_shot_table_column_MMQA_20260303_064042,gpt-5-mini/few_shot_table_column_MMQA_20260303...
6,gpt-5-mini,zero_shot,baseline,zero_shot_baseline_schema_linking_MMQA_2026030...,gpt-5-mini/zero_shot_baseline_schema_linking_M...
7,gpt-5-mini,zero_shot,table_column,zero_shot_table_column_MMQA_20260303_143234,gpt-5-mini/zero_shot_table_column_MMQA_2026030...


In [9]:
evaluation_details: Dict[str, pd.DataFrame] = {}
summary_rows: List[Dict[str, Any]] = []

for prediction_entry in prediction_entries:
    prediction_records: Dict[str, Dict[str, Any]] = load_prediction_records(prediction_entry["path"])

    for dataset_split in DATASET_SPLITS:
        split_ground_truth: Dict[str, Dict[str, Any]] = filter_ground_truth_by_split(ground_truth_records, dataset_split)
        detail_df, table_scores, column_scores = evaluate_predictions(split_ground_truth, prediction_records)

        detail_key: str = f"{prediction_entry['prediction_file']}::{dataset_split}"
        evaluation_details[detail_key] = detail_df
        summary_rows.append(
            {
                "model": prediction_entry["model"],
                "shot_setting": prediction_entry["shot_setting"],
                "method": prediction_entry["method"],
                "prediction_name": prediction_entry["prediction_name"],
                "prediction_file": prediction_entry["prediction_file"],
                "dataset_split": dataset_split,
                "example_count": len(split_ground_truth),
                "table_precision": round(table_scores["precision"], 4),
                "table_recall": round(table_scores["recall"], 4),
                "table_f1": round(table_scores["f1"], 4),
                "column_precision": round(column_scores["precision"], 4),
                "column_recall": round(column_scores["recall"], 4),
                "column_f1": round(column_scores["f1"], 4),
            }
        )

summary_df: pd.DataFrame = pd.DataFrame(summary_rows).sort_values(
    ["dataset_split", "shot_setting", "method", "model", "prediction_file"]
).reset_index(drop=True)


In [17]:
summary_df[["model","shot_setting","method","dataset_split","example_count","table_precision","table_recall","column_precision","column_recall"]]


,model,shot_setting,method,dataset_split,example_count,table_precision,table_recall,column_precision,column_recall
0,gpt-4o-mini,few_shot,baseline,three_table,721,0.5191,0.5518,0.5006,0.4827
1,gpt-5-mini,few_shot,baseline,three_table,721,0.9596,0.9792,0.9402,0.9680
2,gpt-4o-mini,few_shot,table_column,three_table,721,0.9020,0.9536,0.4357,0.4117
3,gpt-5-mini,few_shot,table_column,three_table,721,0.9601,0.9806,0.9420,0.9694
4,gpt-4o-mini,zero_shot,baseline,three_table,721,0.0013,0.0014,0.0013,0.0012
5,gpt-5-mini,zero_shot,baseline,three_table,721,0.9574,0.9778,0.9385,0.9670
6,gpt-4o-mini,zero_shot,table_column,three_table,721,0.9383,0.5035,0.0035,0.0032
7,gpt-5-mini,zero_shot,table_column,three_table,721,0.9587,0.9778,0.9435,0.9666
8,gpt-4o-mini,few_shot,baseline,two_table,2592,0.4457,0.5584,0.4345,0.4856
9,gpt-5-mini,few_shot,baseline,two_table,2592,0.9040,0.9909,0.8742,0.9665


In [11]:
# Optional inspection cell: choose a model / shot setting / method / split, then inspect per-example TP / FP / FN.
selected_model: str = summary_df.iloc[0]["model"]
selected_shot_setting: str = summary_df.iloc[0]["shot_setting"]
selected_method: str = summary_df.iloc[0]["method"]
selected_prediction_file: str = summary_df.loc[
    (summary_df["model"] == selected_model)
    & (summary_df["shot_setting"] == selected_shot_setting)
    & (summary_df["method"] == selected_method),
    "prediction_file",
].iloc[0]
selected_dataset_split: str = "two_table"
evaluation_details[f"{selected_prediction_file}::{selected_dataset_split}"].head(10)


,id,table_tp,table_fp,table_fn,column_tp,column_fp,column_fn
0,MMQA-two_table-0,0,2,2,0,4,5
1,MMQA-two_table-1,2,0,0,5,0,1
2,MMQA-two_table-2,2,1,0,4,1,1
3,MMQA-two_table-3,2,0,0,4,0,1
4,MMQA-two_table-4,0,3,2,0,6,5
5,MMQA-two_table-5,2,0,0,4,1,1
6,MMQA-two_table-6,0,2,2,0,7,6
7,MMQA-two_table-7,2,1,0,5,1,0
8,MMQA-two_table-8,0,2,2,0,4,4
9,MMQA-two_table-9,2,0,0,4,2,0
